In [3]:
from process_mobidb_scores import get_mobidb_scores
from alphasync_data import get_alphasync_data, parse_alphasync_data
from protein_region_features import features_binding_region_df, get_amino_acid_composition, calculate_scd, calculate_shd, calculate_fcr, get_sec_str_freq, get_average_feature_value
from pathlib import Path
import pandas as pd
import numpy as np

In [4]:
# get the current directory
current_dir = Path.cwd()

# path to the input fasta file
input_fasta = current_dir.parents[1] / "data" / "raw" / "mobidb_search_2026-05-05T08-28-46.fasta"

# path to the processed data folder
processed_data_folder = current_dir.parents[1] / "data" / "processed"

# define the output data folder for the alphasync data
output_data_folder = current_dir.parents[1] / "data" / "raw" / "alphasync_data"



In [5]:
# get the mobidb scores for the given input fasta file and the given score patterns
human_proteome_seq_df = get_mobidb_scores(str(input_fasta), ["derived-binding_mode_disorder_to_disorder-mobi", "derived-binding_mode_disorder_to_order-mobi", "derived-binding_mode_context_dependent-mobi"])

# print the number of rows in the complete data dataframe
print(f"Number of rows in the complete data dataframe: {human_proteome_seq_df.shape[0]}")

Number of rows in the complete data dataframe: 82518


In [6]:
# filter the dataframe to get only those that has mobidb scores for at least one of the binding mode scores
binding_mode_scores = ["derived_binding_mode_disorder_to_disorder_mobi", "derived_binding_mode_disorder_to_order_mobi", "derived_binding_mode_context_dependent_mobi"]
df_binding_idr_seq = human_proteome_seq_df.dropna(subset=binding_mode_scores, how="all").copy()

# print the number of rows in the filtered dataframe
print(f"Number of rows in the filtered dataframe: {df_binding_idr_seq.shape[0]}")

# save the dataframe as csv files in the processed data folder
df_binding_idr_seq.to_csv(processed_data_folder / "binding_idr_sequences.csv", index=False)

Number of rows in the filtered dataframe: 4751


In [7]:
# get the alphasync data for the binding idr sequences
print("Downloading the alphasync data for the given uniprot IDs...")
get_alphasync_data(df_binding_idr_seq["uniprot_id"].tolist(), str(output_data_folder))

Failed to download E9PG32 (Status: 404)
Failed to download A0A8V8TRG9 (Status: 404)
Failed to download A0A0A0MSA1 (Status: 404)
Failed to download F2Z2U4 (Status: 404)
Failed to download A0A669KB38 (Status: 404)


In [8]:
# remove the ids that couldn't be downloaded from the alphasync data and save the filtered dataframe as a csv file in the processed data folder
remove_ids = ["A0A0A0MSA1", "F2Z2U4", "E9PG32", "A0A8V8TRG9", "A0A669KB38"]

# filter the dataframe to remove the rows with the uniprot ids in the remove_ids list
df_binding_idr_seq_filtered = df_binding_idr_seq[~df_binding_idr_seq["uniprot_id"].isin(remove_ids)].copy()

In [9]:
# add the alphasync data for the filtered dataframe
df_binding_idr_seq_filtered["plddt"] = df_binding_idr_seq_filtered["uniprot_id"].apply(lambda x: parse_alphasync_data(x, "plddt", str(output_data_folder), divide_by_100=True))
df_binding_idr_seq_filtered["relasa"] = df_binding_idr_seq_filtered["uniprot_id"].apply(lambda x: parse_alphasync_data(x, "relasa", str(output_data_folder)))
df_binding_idr_seq_filtered["sec"] = df_binding_idr_seq_filtered["uniprot_id"].apply(lambda x: parse_alphasync_data(x, "sec", str(output_data_folder)))

print(df_binding_idr_seq_filtered.head())

  uniprot_id                                           sequence  \
1     Q04917  MGDREQLLQRARLAEQAERYDDMASAMKAVTELNEPLSNEDRNLLS...   
2     P62258  MDDREDLVYQAKLAEQAERYDEMVESMKKVAGMDVELTVEERNLLS...   
3     P61981  MVDREQLVQKARLAEQAERYDDMAAAMKNVTELNEPLSNEERNLLS...   
4     P31947  MERASLIQKAKLAEQAERYEDMAAFMKGAVEKGEELSCEERNLLSV...   
5     P27348  MEKTELIQKAKLAEQAERYDDMATCMKAVTEQGAELSNEERNLLSV...   

      derived_binding_mode_disorder_to_disorder_mobi  \
1  0000000000000000000000000000000000000000000000...   
2  0000000000000000000000000000000000000000000000...   
3                                                NaN   
4  0000000000000000000000000000000000000000000000...   
5  0000000000000000000000000000000000000000000000...   

         derived_binding_mode_disorder_to_order_mobi  \
1                                                NaN   
2  1111111111111111111111111111111100000000000000...   
3  0000000000000000000000000000000000000000000000...   
4                                   

In [10]:

# get features for the binding regions from that of the complete protein sequence
df_binding_region_do = features_binding_region_df(df_binding_idr_seq_filtered, "derived_binding_mode_disorder_to_order_mobi", {"plddt", "relasa", "sec"})
df_binding_region_do["binding_mode"] = "disorder_to_order"
print(df_binding_region_do.head())
df_binding_region_dd = features_binding_region_df(df_binding_idr_seq_filtered, "derived_binding_mode_disorder_to_disorder_mobi", {"plddt", "relasa", "sec"})
df_binding_region_dd["binding_mode"] = "disorder_to_disorder"
print(df_binding_region_dd.head())
df_binding_region_cd = features_binding_region_df(df_binding_idr_seq_filtered, "derived_binding_mode_context_dependent_mobi", {"plddt", "relasa", "sec"})
df_binding_region_cd["binding_mode"] = "context_dependent"
print(df_binding_region_cd.head())


# save the dataframes as csv files and pickle files in the processed data folder
df_binding_region_do.to_csv(processed_data_folder / "binding_region_disorder_to_order.csv", index=False)
df_binding_region_dd.to_csv(processed_data_folder / "binding_region_disorder_to_disorder.csv", index=False)
df_binding_region_cd.to_csv(processed_data_folder / "binding_region_context_dependent.csv", index=False)

  uniprot_id                            binding_region_seq  start_pos  \
0     P62258              MDDREDLVYQAKLAEQAERYDEMVESMKKVAG          0   
1     P61981                                FDDAIAELDTLNED        200   
2     Q15172                            QAELHPLPQLKDATSNEQ         49   
3     Q15172                         LFDDLTSSYKAERQREKKKEL        428   
4     Q14738  NSTPPPTQLSKIKYSGGPQIVKKERRQSSSRFNLSKNRELQKLP         60   

   end_pos                                             relasa  \
0       32  [0.8867, 0.5401, 0.5562, 0.1283, 0.5467, 0.320...   
1      214  [0.1491, 0.3636, 0.3476, 0.0, 0.441, 0.6942, 0...   
2       67  [0.7336, 0.4545, 0.5, 0.0366, 0.588, 0.5974, 0...   
3      449  [0.178, 0.0965, 0.5187, 0.492, 0.1675, 0.0982,...   
4      104  [0.8021, 0.8811, 0.7055, 0.7922, 0.7078, 0.798...   

                                               plddt  \
0  [0.5348, 0.8042, 0.9209, 0.9713, 0.97, 0.9722,...   
1  [0.9804, 0.9806, 0.9766, 0.9707, 0.9641, 0.964...   
2 

In [11]:
# combine the three dataframes into one dataframe
df_binding_region_combined = pd.concat([df_binding_region_do, df_binding_region_dd, df_binding_region_cd], ignore_index=True)

# filter the dataframe to remove rows where the plddt column is empty or not a list
is_not_empty = df_binding_region_combined["plddt"].apply(lambda x: isinstance(x, list) and len(x) > 0)
df_binding_region_combined = df_binding_region_combined[is_not_empty]

# filter the dataframe to remove same binding_region_seq
df_binding_region_combined = df_binding_region_combined.drop_duplicates(subset=["binding_region_seq"])

In [12]:
# add the features for the binding regions to the combined dataframe
df_binding_region_combined["amino_acid_composition"] = df_binding_region_combined["binding_region_seq"].apply(get_amino_acid_composition)
df_binding_region_combined["scd"] = df_binding_region_combined["binding_region_seq"].apply(lambda x: calculate_scd(x, need_rounding=True, round_to=4))
df_binding_region_combined["shd"] = df_binding_region_combined["binding_region_seq"].apply(lambda x: calculate_shd(x, need_rounding=True, round_to=4))
df_binding_region_combined["fcr"] = df_binding_region_combined["binding_region_seq"].apply(lambda x: calculate_fcr(x, need_rounding=True, round_to=4))
df_binding_region_combined["sec_str_freq"] = df_binding_region_combined["sec"].apply(get_sec_str_freq)
df_binding_region_combined["sec_str3_freq"] = df_binding_region_combined["sec"].apply(lambda x: get_sec_str_freq(x, convert_3_state=True))
df_binding_region_combined["avg_plddt"] = df_binding_region_combined["plddt"].map(np.mean).round(4)
df_binding_region_combined["avg_relasa"] = df_binding_region_combined["relasa"].map(np.mean).round(4)

# print the first 5 rows of the combined dataframe with features
print(df_binding_region_combined.head())

# save the combined dataframe with features as a csv file in the processed data folder
df_binding_region_combined.to_csv(processed_data_folder / "binding_region_combined_features.csv", index=False)

  uniprot_id                            binding_region_seq  start_pos  \
0     P62258              MDDREDLVYQAKLAEQAERYDEMVESMKKVAG          0   
1     P61981                                FDDAIAELDTLNED        200   
2     Q15172                            QAELHPLPQLKDATSNEQ         49   
3     Q15172                         LFDDLTSSYKAERQREKKKEL        428   
4     Q14738  NSTPPPTQLSKIKYSGGPQIVKKERRQSSSRFNLSKNRELQKLP         60   

   end_pos                                             relasa  \
0       32  [0.8867, 0.5401, 0.5562, 0.1283, 0.5467, 0.320...   
1      214  [0.1491, 0.3636, 0.3476, 0.0, 0.441, 0.6942, 0...   
2       67  [0.7336, 0.4545, 0.5, 0.0366, 0.588, 0.5974, 0...   
3      449  [0.178, 0.0965, 0.5187, 0.492, 0.1675, 0.0982,...   
4      104  [0.8021, 0.8811, 0.7055, 0.7922, 0.7078, 0.798...   

                                               plddt  \
0  [0.5348, 0.8042, 0.9209, 0.9713, 0.97, 0.9722,...   
1  [0.9804, 0.9806, 0.9766, 0.9707, 0.9641, 0.964...   
2 

In [13]:
# expand the features
df_aa = pd.DataFrame(df_binding_region_combined["amino_acid_composition"].tolist(), index=df_binding_region_combined.index).add_prefix("aa_freq_")
df_ss = pd.DataFrame(df_binding_region_combined["sec_str_freq"].tolist(), index=df_binding_region_combined.index).add_prefix("ss_freq_")
df_ss3 = pd.DataFrame(df_binding_region_combined["sec_str3_freq"].tolist(), index=df_binding_region_combined.index).add_prefix("ss3_freq_")
df_final = pd.concat([df_binding_region_combined.drop(columns=["amino_acid_composition", "sec_str_freq", "sec_str3_freq"]), df_aa, df_ss, df_ss3], axis=1)

print(df_final.head())

# save the final dataframe with expanded features as a pickle file in the processed data folder
df_final.to_pickle(processed_data_folder / "binding_region_combined_features_expanded.pkl")


  uniprot_id                            binding_region_seq  start_pos  \
0     P62258              MDDREDLVYQAKLAEQAERYDEMVESMKKVAG          0   
1     P61981                                FDDAIAELDTLNED        200   
2     Q15172                            QAELHPLPQLKDATSNEQ         49   
3     Q15172                         LFDDLTSSYKAERQREKKKEL        428   
4     Q14738  NSTPPPTQLSKIKYSGGPQIVKKERRQSSSRFNLSKNRELQKLP         60   

   end_pos                                             relasa  \
0       32  [0.8867, 0.5401, 0.5562, 0.1283, 0.5467, 0.320...   
1      214  [0.1491, 0.3636, 0.3476, 0.0, 0.441, 0.6942, 0...   
2       67  [0.7336, 0.4545, 0.5, 0.0366, 0.588, 0.5974, 0...   
3      449  [0.178, 0.0965, 0.5187, 0.492, 0.1675, 0.0982,...   
4      104  [0.8021, 0.8811, 0.7055, 0.7922, 0.7078, 0.798...   

                                               plddt  \
0  [0.5348, 0.8042, 0.9209, 0.9713, 0.97, 0.9722,...   
1  [0.9804, 0.9806, 0.9766, 0.9707, 0.9641, 0.964...   
2 